# Phase 4 — Data Preprocessing & Feature Engineering

This phase converts the validated `merged_data` into a machine-learning-ready representation while preventing data leakage.

**Important:** The train/test split happens before any learned preprocessing. No SMOTE, model training, threshold tuning, SHAP, API, or database work is performed.

## 1. Preprocessing Objective

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

print("Objective: split first, fit transformations on X_train only, then transform train and test.")
print("SMOTE and model training: NOT PERFORMED")

Objective: split first, fit transformations on X_train only, then transform train and test.
SMOTE and model training: NOT PERFORMED


## 2. Load merged_data

In [2]:
project = Path.cwd().parent
merged_path = project / "data" / "processed" / "merged_data.csv.gz"
merged_data = pd.read_csv(merged_path, compression="gzip", low_memory=False)
print("Shape:", merged_data.shape)
display(merged_data.head())

Shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## 3. Initial Validation

In [3]:
print("Data types:")
display(merged_data.dtypes.value_counts())
print("Total missing cells:", int(merged_data.isna().sum().sum()))
print("Target exists:", "isFraud" in merged_data.columns)
assert "isFraud" in merged_data.columns
assert merged_data.shape == (590540, 434)

Data types:


float64    399
str         31
int64        4
Name: count, dtype: int64

Total missing cells: 115523073
Target exists: True


**Interpretation:** The dimensions match the validated Phase 2 output. The source data contains missing values that must be handled after the split.

## 4. Separate Features and Target

In [4]:
X = merged_data.drop("isFraud", axis=1).copy()
y = merged_data["isFraud"].copy()

# TransactionID is a direct identifier and is excluded from model inputs.
excluded_features = pd.DataFrame({
    "Feature": ["TransactionID"],
    "Reason for Exclusion": ["Direct transaction identifier; retaining it may allow memorization rather than generalizable behavior."]
})
X = X.drop("TransactionID", axis=1)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Fraud:", int((y == 1).sum()))
print("Legitimate:", int((y == 0).sum()))
display(excluded_features)

X shape: (590540, 432)
y shape: (590540,)
Fraud: 20663
Legitimate: 569877


,Feature,Reason for Exclusion
0,TransactionID,Direct transaction identifier; retaining it ma...


## 5. Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training fraud %:", y_train.mean() * 100)
print("Testing fraud %:", y_test.mean() * 100)

Training rows: 472432
Testing rows: 118108
Training fraud %: 3.498916246147594
Testing fraud %: 3.49933958749619


**Why split first?** The split is performed before fitting imputers, encoders, frequency mappings, scalers, or feature engineering decisions that learn from data. This prevents test information from influencing training transformations.

## 6. Identify Numerical and Categorical Features

In [6]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("First numerical features:", numeric_features[:10])
print("Categorical features:", categorical_features)

Numerical features: 401
Categorical features: 31
First numerical features: ['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2']
Categorical features: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


C:\Users\HP\AppData\Local\Temp\ipykernel_3112\1334022351.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 7. Feature Decisions Based on EDA

In [7]:
print("EDA Finding → Decision → Reason")
print("High missingness → median for numeric and explicit Missing for categorical → retain rows and avoid arbitrary deletion")
print("High/medium categorical cardinality → frequency encoding → avoid a very wide one-hot matrix")
print("Low categorical cardinality → one-hot encoding → preserve category information safely")
print("Identity availability association → selected missingness indicators → allow the model to inspect availability explicitly")
print("TransactionID → exclude → direct identifier and potential memorization risk")

EDA Finding → Decision → Reason
High missingness → median for numeric and explicit Missing for categorical → retain rows and avoid arbitrary deletion
High/medium categorical cardinality → frequency encoding → avoid a very wide one-hot matrix
Low categorical cardinality → one-hot encoding → preserve category information safely
Identity availability association → selected missingness indicators → allow the model to inspect availability explicitly
TransactionID → exclude → direct identifier and potential memorization risk


## 8. Missing Value and Cardinality Strategy

In [8]:
cardinality_table = pd.DataFrame({
    "Feature": categorical_features,
    "Unique Values": [X_train[c].nunique(dropna=False) for c in categorical_features]
})
cardinality_table["Category"] = np.where(cardinality_table["Unique Values"] <= 10, "Low", "High/Medium")
low_cardinality_features = cardinality_table.loc[cardinality_table["Category"] == "Low", "Feature"].tolist()
high_cardinality_features = cardinality_table.loc[cardinality_table["Category"] == "High/Medium", "Feature"].tolist()
print("Low-cardinality features:", len(low_cardinality_features))
print("High/medium-cardinality features:", len(high_cardinality_features))
display(cardinality_table.sort_values("Unique Values", ascending=False))

Low-cardinality features: 25
High/medium-cardinality features: 6


,Feature,Unique Values,Category
30,DeviceInfo,1687,High/Medium
23,id_33,237,High/Medium
22,id_31,125,High/Medium
21,id_30,76,High/Medium
4,R_emaildomain,61,High/Medium
3,P_emaildomain,60,High/Medium
0,ProductCD,5,Low
2,card6,5,Low
1,card4,5,Low
24,id_34,5,Low


Numerical missing values use **median imputation** because transaction variables can be skewed. Categorical missing values become the explicit category **`Missing`**. Both transformations are fitted on `X_train` only.

## 9. Simple Feature Engineering

In [9]:
# These indicators use only whether a value is missing; they do not learn from test data.
engineered_features = []
for feature in ["DeviceInfo", "id_33"]:
    if feature in X_train.columns:
        new_feature = feature + "_is_missing"
        X_train[new_feature] = X_train[feature].isna().astype("int8")
        X_test[new_feature] = X_test[feature].isna().astype("int8")
        engineered_features.append(new_feature)

print("Created features:", engineered_features)
print("Reason: EDA showed partial identity coverage and meaningful missingness patterns.")
# Re-identify columns after adding indicators.
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
cardinality_table = pd.DataFrame({"Feature": categorical_features, "Unique Values": [X_train[c].nunique(dropna=False) for c in categorical_features]})
cardinality_table["Category"] = np.where(cardinality_table["Unique Values"] <= 10, "Low", "High/Medium")
low_cardinality_features = cardinality_table.loc[cardinality_table["Category"] == "Low", "Feature"].tolist()
high_cardinality_features = cardinality_table.loc[cardinality_table["Category"] == "High/Medium", "Feature"].tolist()

Created features: ['DeviceInfo_is_missing', 'id_33_is_missing']
Reason: EDA showed partial identity coverage and meaningful missingness patterns.


C:\Users\HP\AppData\Local\Temp\ipykernel_3112\1330816894.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


**Feature engineering decision:** Only two simple missingness indicators were created when the fields were available. No artificial calendar timestamps, target-derived variables, or large feature families were created.

## 10. Build the Preprocessing Pipeline

In [10]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Learn one frequency map per column from training data only."""
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.maps_ = []
        for column in X.columns:
            frequencies = X[column].value_counts(normalize=True, dropna=False).to_dict()
            self.maps_.append(frequencies)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        result = pd.DataFrame(index=X.index)
        for i, column in enumerate(X.columns):
            result[column] = X[column].map(self.maps_[i]).fillna(0).astype("float32")
        return result.to_numpy()

def to_sparse(X):
    return sparse.csr_matrix(X)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
    ("sparse", FunctionTransformer(to_sparse, accept_sparse=True))
])

low_cardinality_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32))
])

high_cardinality_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("frequency", FrequencyEncoder()),
    ("sparse", FunctionTransformer(to_sparse, accept_sparse=True))
])

transformers = [("numeric", numeric_pipeline, numeric_features)]
if low_cardinality_features:
    transformers.append(("low_cardinality", low_cardinality_pipeline, low_cardinality_features))
if high_cardinality_features:
    transformers.append(("high_cardinality", high_cardinality_pipeline, high_cardinality_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=1.0)
print("Pipeline created with numeric, low-cardinality, and high/medium-cardinality branches.")

Pipeline created with numeric, low-cardinality, and high/medium-cardinality branches.


## 11. Fit Only on Training Data and Transform Both Sets

In [11]:
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)
print("Original X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)
print("Processed representation:", type(X_train_processed).__name__)
print("Processed non-zero values in train:", X_train_processed.nnz if sparse.issparse(X_train_processed) else np.count_nonzero(X_train_processed))

Original X_train shape: (472432, 434)
Processed X_train shape: (472432, 495)
Original X_test shape: (118108, 434)
Processed X_test shape: (118108, 495)
Processed representation: csr_matrix
Processed non-zero values in train: 103245964


**Interpretation:** One-hot encoding increases the feature count for low-cardinality categories, while frequency encoding keeps high/medium-cardinality categories compact. Scaling was fitted within the numeric branch using training data only.

## 12. Validate Processed Data

In [12]:
def check_processed(name, data):
    values = data.data if sparse.issparse(data) else data
    print(name, "is numeric:", np.issubdtype(values.dtype, np.number))
    print(name, "NaN count:", int(np.isnan(values).sum()))
    print(name, "Inf count:", int(np.isinf(values).sum()))

check_processed("X_train_processed", X_train_processed)
check_processed("X_test_processed", X_test_processed)
print("Same number of features:", X_train_processed.shape[1] == X_test_processed.shape[1])
print("Same feature order: True (same fitted ColumnTransformer)")

X_train_processed is numeric: True


X_train_processed NaN count: 0


X_train_processed Inf count: 0
X_test_processed is numeric: True
X_test_processed NaN count: 0
X_test_processed Inf count: 0
Same number of features: True
Same feature order: True (same fitted ColumnTransformer)


## 13. Verify Stratification and Leakage Controls

In [13]:
leakage_check = pd.DataFrame({
    "Operation": ["Median imputation", "One-hot categories", "Frequency encoding", "Scaling", "Feature engineering"],
    "Train Only?": ["Yes", "Yes", "Yes", "Yes", "Yes"],
    "Leakage Risk": ["No", "No", "No", "No", "No"]
})
display(leakage_check)
print("Train fraud %:", y_train.mean() * 100)
print("Test fraud %:", y_test.mean() * 100)
print("Split was stratified: True")
print("Frequency mappings were learned in FrequencyEncoder.fit(X_train): True")
print("Test categories unseen during training use frequency fallback 0 or one-hot ignore: True")

,Operation,Train Only?,Leakage Risk
0,Median imputation,Yes,No
1,One-hot categories,Yes,No
2,Frequency encoding,Yes,No
3,Scaling,Yes,No
4,Feature engineering,Yes,No


Train fraud %: 3.498916246147594
Test fraud %: 3.49933958749619
Split was stratified: True
Frequency mappings were learned in FrequencyEncoder.fit(X_train): True
Test categories unseen during training use frequency fallback 0 or one-hot ignore: True


## 14. Save the Preprocessor and Appropriate Processed Data

In [14]:
models_folder = project / "models"
processed_folder = project / "data" / "processed"
models_folder.mkdir(exist_ok=True)
processed_folder.mkdir(exist_ok=True)

preprocessor_path = models_folder / "preprocessor.pkl"
joblib.dump(preprocessor, preprocessor_path)
print("Saved preprocessor:", preprocessor_path)

# Save sparse matrices and targets instead of creating large dense duplicate CSV files.
sparse.save_npz(processed_folder / "X_train_processed.npz", X_train_processed)
sparse.save_npz(processed_folder / "X_test_processed.npz", X_test_processed)
y_train.to_csv(processed_folder / "y_train.csv", index=False)
y_test.to_csv(processed_folder / "y_test.csv", index=False)
print("Saved sparse processed train/test matrices and target files.")

Saved preprocessor: C:\Users\HP\Desktop\fraud-detection-system\models\preprocessor.pkl


Saved sparse processed train/test matrices and target files.


## 15. Phase 4 — Final Preprocessing Report

In [15]:
print("PHASE 4 — FINAL PREPROCESSING REPORT")
print("=" * 45)
print("Original Rows:", len(merged_data))
print("Original Columns:", len(merged_data.columns))
print("Train Rows:", len(X_train))
print("Test Rows:", len(X_test))
print("Train Fraud %:", y_train.mean() * 100)
print("Test Fraud %:", y_test.mean() * 100)
print("Numerical Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))
print("High/Medium Cardinality Features:", len(high_cardinality_features))
print("Low Cardinality Features:", len(low_cardinality_features))
print("Numerical Strategy: median imputation fitted on X_train, then StandardScaler fitted on X_train")
print("Categorical Strategy: Missing category fitted on X_train")
print("Low Cardinality Encoding: OneHotEncoder(handle_unknown='ignore')")
print("High/Medium Cardinality Encoding: frequency encoding learned on X_train")
print("Feature Engineering:", engineered_features)
print("Before preprocessing:", X_train.shape, X_test.shape)
print("After preprocessing:", X_train_processed.shape, X_test_processed.shape)
print("No unexpected NaN or Inf: True")
print("Train/Test dimensions match: True")
print("Preprocessing fitted only on Train: True")
print("Preprocessor artifact:", preprocessor_path)
print("Processed data artifacts: sparse NPZ matrices and y CSV files")
print("Model training: NOT PERFORMED")
print("Decision: Processed data is ready for Phase 5 baseline modeling after review.")

PHASE 4 — FINAL PREPROCESSING REPORT
Original Rows: 590540
Original Columns: 434
Train Rows: 472432
Test Rows: 118108
Train Fraud %: 3.498916246147594
Test Fraud %: 3.49933958749619
Numerical Features: 403
Categorical Features: 31
High/Medium Cardinality Features: 6
Low Cardinality Features: 25
Numerical Strategy: median imputation fitted on X_train, then StandardScaler fitted on X_train
Categorical Strategy: Missing category fitted on X_train
Low Cardinality Encoding: OneHotEncoder(handle_unknown='ignore')
High/Medium Cardinality Encoding: frequency encoding learned on X_train
Feature Engineering: ['DeviceInfo_is_missing', 'id_33_is_missing']
Before preprocessing: (472432, 434) (118108, 434)
After preprocessing: (472432, 495) (118108, 495)
No unexpected NaN or Inf: True
Train/Test dimensions match: True
Preprocessing fitted only on Train: True
Preprocessor artifact: C:\Users\HP\Desktop\fraud-detection-system\models\preprocessor.pkl
Processed data artifacts: sparse NPZ matrices and y C

**STOP:** Phase 4 is complete. Do not start `05_baseline_model.ipynb` or train any model until explicit approval.